# String vectorisation
This notebook shows how to vectorise strings using OpenAI's embedding API, using following models:
- text-embedding-3-large
- text-embedding-3-small

Let's import the necessary libraries:

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI

from qdrant_client import QdrantClient
from qdrant_client.models import Distance, PointStruct, VectorParams

load_dotenv("../../../_starter/.env")

True

# Initialize the client

In [2]:
client = OpenAI(
    base_url = os.getenv("AZURE_COGNITIVE_ENDPOINT"),
    api_key = os.getenv("AZURE_COGNITIVE_KEY"),
)

# How to vectorise strings
We can vectorise strings using OpenAI's embedding API through Azure OpenAI or OpenAI's API.

To vectorise something we need two things:
- string to vectorise
- embedding model

Let's vectorise "apple" using "text-embedding-3-large" model.

In [3]:
string_to_vectorise = "apple"

deployment_name = "text-embedding-3-large"

# Vectorise the string using OpenAI's embedding API
response = client.embeddings.create(
    input = "How do I use Python in VS Code?",
    model = deployment_name
)
vector = response.data[0].embedding

# Let's print first 5 elements of the vector
print(vector[0:5])

[-0.008977994322776794, -0.007912185974419117, -0.022471237927675247, 0.02929661236703396, -0.017262950539588928]


# How to make vectors for a small dataset

Let's create vectors for a small dataset of 20 random foods and items.

In [4]:
dataset = [
    "apple",
    "banana",
    "orange",
    "pear",
    "pineapple",
    "joghurt",
    "pizza",
    "car",
    "house",
    "tree",
    "flower",
    "book",
    "computer",
    "phone",
    "laptop",
    "mouse",
    "keyboard",
    "monitor",
    "printer",
    "chair",
]

This time we will do this in a batch, so instead of vectorising one string at a time, we will vectorise all of them at once.

If you have a larger dataset, you might want to do this in batches, to avoid hitting the API limit.

In [5]:
response = client.embeddings.create(
    input = dataset,
    model = deployment_name
)

vectors = response.data

# Let's print first 5 elements of the first 3 vectors
print(vectors[0].embedding[0:5])
print(vectors[1].embedding[0:5])
print(vectors[2].embedding[0:5])

[-0.020793478935956955, 0.013975388370454311, -0.0008183403988368809, 0.018724307417869568, -0.008179164491593838]
[-0.007053900044411421, -0.009362380020320415, -0.006324707064777613, 0.02643228881061077, -0.030361616984009743]
[-0.02574823796749115, 0.004719946533441544, -0.004284324124455452, -0.009600607678294182, -0.024665527045726776]


Let's create a dictionary of vectors, so we can easily access them by the original string.

In [6]:
vectors_dict = {item: vector.embedding for item, vector in zip(dataset, vectors)}

# Let's print first 5 elements of the vector for "apple"
print(vectors_dict["house"][0:5])

[0.006906401365995407, 0.0017256085993722081, -0.01837475597858429, -0.0031120458152145147, 0.007771189324557781]


Let' create a new Qdrant collection and add our vectors to it, so we can use them for similarity search in the next example.

In [7]:
qdrant_client = QdrantClient(url="http://localhost:6333")

dimensions = 3072 # Make sure the dimension count matches the embedding model you are using, is this case text-embedding-3-large

# Create the collection
qdrant_client.create_collection(
    collection_name="my_random_items",
    vectors_config=VectorParams(size=dimensions, distance=Distance.COSINE),
)

# Add the vectors to the collection
for idx, (item, vector) in enumerate(vectors_dict.items()):
    qdrant_client.upsert(
        collection_name="my_random_items",
        points=[PointStruct(id=idx, vector=vector, payload={"text": item})],
    )

# Let's check how many points we have in the collection
qdrant_client.get_collection("my_random_items").points_count

20